<a href="https://colab.research.google.com/github/AbinayaUA/DEEPSEEK_OCR-2_Medical-Prescription-Reader/blob/main/Medical_Prescription_Reader_DEEPSEEK_OCR_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#  Install required system fonts for bounding box rendering
!apt-get update
!apt-get install -y fonts-dejavu-core

#  Install the required Python packages
!pip install torch torchvision transformers==4.46.3 tokenizers==0.20.3 accelerate einops addict easydict PyMuPDF hf_transfer spaces gradio

Cloning into 'DeepSeek-OCR-Demo'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 172 (delta 3), reused 0 (delta 0), pack-reused 163 (from 1)
Receiving objects: 100% (172/172), 60.93 KiB | 3.58 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/DeepSeek-OCR-Demo
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 http://archive.ubuntu.com/ub

In [10]:
import gradio as gr
from transformers import AutoModel, AutoTokenizer
import torch
import spaces
import os
import sys
import tempfile
import shutil
from PIL import Image, ImageDraw, ImageFont, ImageOps
import re
import numpy as np
from io import StringIO

# --- MODEL INITIALIZATION ---
MODEL_NAME = 'deepseek-ai/DeepSeek-OCR-2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    use_safetensors=True
)
model = model.eval().cuda()

BASE_SIZE = 1024
IMAGE_SIZE = 768
CROP_MODE = True

# --- HELPER FUNCTIONS ---
def extract_grounding_references(text):
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    return re.findall(pattern, text, re.DOTALL)

def draw_bounding_boxes(image, refs):
    img_w, img_h = image.size
    img_draw = image.copy()
    draw = ImageDraw.Draw(img_draw)
    overlay = Image.new('RGBA', img_draw.size, (0, 0, 0, 0))
    draw2 = ImageDraw.Draw(overlay)

    # Fallback to default font if Dejavu is missing in your environment
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 15)
    except:
        font = ImageFont.load_default()

    color_map = {}
    np.random.seed(42)

    for ref in refs:
        label = ref[1]
        if label not in color_map:
            color_map[label] = (np.random.randint(50, 255), np.random.randint(50, 255), np.random.randint(50, 255))

        color = color_map[label]
        coords = eval(ref[2])
        color_a = color + (60,)

        for box in coords:
            x1, y1, x2, y2 = int(box[0]/999*img_w), int(box[1]/999*img_h), int(box[2]/999*img_w), int(box[3]/999*img_h)

            width = 5 if label == 'title' else 3
            draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
            draw2.rectangle([x1, y1, x2, y2], fill=color_a)

            text_bbox = draw.textbbox((0, 0), label, font=font)
            tw, th = text_bbox[2] - text_bbox[0], text_bbox[3] - text_bbox[1]
            ty = max(0, y1 - 20)
            draw.rectangle([x1, ty, x1 + tw + 4, ty + th + 4], fill=color)
            draw.text((x1 + 2, ty + 2), label, font=font, fill=(255, 255, 255))

    img_draw.paste(overlay, (0, 0), overlay)
    return img_draw

def clean_output(text):
    if not text:
        return ""

    # Completely remove the tracking tags (<|ref|>...<|/det|>)
    pattern = r'<\|ref\|>.*?<\|/ref\|><\|det\|>.*?<\|/det\|>'
    text = re.sub(pattern, '', text, flags=re.DOTALL)

    text = text.replace('\\coloneqq', ':=').replace('\\eqqcolon', '=:')

    # Clean up any lingering blank lines or spaces
    lines = [line.strip() for line in text.split('\n') if line.strip()]

    return '\n'.join(lines)

# --- MAIN INFERENCE LOGIC ---
@spaces.GPU(duration=90)
def analyze_prescription(image_path):
    if not image_path:
        return "Error: Upload an image", None, ""

    image = Image.open(image_path)
    if image.mode in ('RGBA', 'LA', 'P'):
        image = image.convert('RGB')
    image = ImageOps.exif_transpose(image)

   #  PROMPT: Ban HTML tables and force a literal, line-by-line transcription
    prompt = "<image>\n<|grounding|>Transcribe all handwritten and printed text exactly as it appears line by line. Do NOT generate HTML, tables, or markdown formatting. Output plain text only."

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.jpg')
    image.save(tmp.name, 'JPEG', quality=95)
    tmp.close()
    out_dir = tempfile.mkdtemp()

    # Capture stdout as a fallback for Gradio threading
    stdout = sys.stdout
    sys.stdout = StringIO()

    # Capture the direct return value from the model
    output_data = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=tmp.name,
        output_path=out_dir,
        base_size=BASE_SIZE,
        image_size=IMAGE_SIZE,
        crop_mode=CROP_MODE,
        save_results=False
    )

    sys_stdout_val = sys.stdout.getvalue()
    sys.stdout = stdout # Restore stdout immediately

    # Extract the text safely regardless of how the remote model returns it
    raw_result = ""
    if isinstance(output_data, str) and output_data.strip():
        raw_result = output_data
    elif isinstance(output_data, dict) and 'text' in output_data:
        raw_result = output_data['text']
    else:
        # Fallback to the stdout capture if the model returned None
        debug_filters = ['PATCHES', '====', 'BASE:', 'directly resize', 'NO PATCHES', 'torch.Size', '%|']
        raw_result = '\n'.join([l for l in sys_stdout_val.split('\n')
                            if l.strip() and not any(s in l for s in debug_filters)]).strip()

    os.unlink(tmp.name)
    shutil.rmtree(out_dir, ignore_errors=True)

    if not raw_result:
        return "Model returned empty text. Try a clearer image or a different prompt.", image, ""

    # Clean the raw output to generate the final text
    extracted_text = clean_output(raw_result)

    # Draw boxes if grounding tags exist
    img_out = image.copy()
    if '<|ref|>' in raw_result:
        refs = extract_grounding_references(raw_result)
        if refs:
            img_out = draw_bounding_boxes(image, refs)

    return extracted_text, img_out, raw_result


# --- GRADIO 6.0 UI LAYOUT ---
with gr.Blocks(title="Medical Prescription Reader") as demo:
    gr.Markdown("""
    # 🩺 Medical Prescription Reader
    Upload a handwritten medical prescription to extract the text and locate medicines and dosages.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(label="Upload Prescription Image", type="filepath", height=400)
            btn = gr.Button("Analyze Prescription", variant="primary", size="lg")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("Extracted Text"):
                    text_out = gr.Textbox(lines=15, show_label=False)
                with gr.Tab("Bounding Boxes"):
                    img_out = gr.Image(type="pil", height=500, show_label=False)
                with gr.Tab("Raw Output"):
                    raw_out = gr.Textbox(lines=15, show_label=False)

    btn.click(
        fn=analyze_prescription,
        inputs=[input_img],
        outputs=[text_out, img_out, raw_out]
    )

if __name__ == "__main__":
    demo.queue(max_size=20).launch(theme=gr.themes.Soft(primary_hue="teal"))

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14cd3ebfe33c5a4b44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
